In [1]:
import pandas as pd
import numpy as np

## 3. Create hourly aggregated Dateframe
Create a new df with hourly timestamps in the range of the dataset and all unique H3 indices so we have a complete grid for analysis

In [2]:
# TODO: Validate data set paths
taxi_data_processed = pd.read_parquet('../data/processed/taxi_data_processed.parquet')
poi_data_processed = pd.read_csv('../data/processed/chicago_pois_agg.csv')
# weather_data_processed = pd.read_csv('../data/processed/weather_data_processed.csv')

In [3]:
# Extract hour from Trip Start Timestamp and Trip End Timestamp for temporal analysis
taxi_data_processed['Pickup Hour'] = pd.to_datetime(taxi_data_processed['Trip Start Timestamp']).dt.floor('h')
taxi_data_processed['Dropoff Hour'] = pd.to_datetime(taxi_data_processed['Trip End Timestamp']).dt.floor('h')

# Get the range of timestamps in the dataset
min_timestamp = taxi_data_processed['Pickup Hour'].min()
max_timestamp = taxi_data_processed['Dropoff Hour'].max()

# Create a complete hourly timestamp range
hourly_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='h')

# Get all unique H3 indices from both pickup and dropoff
unique_h3_indices_8 = pd.unique(pd.concat([taxi_data_processed['h3_index_pickup_8'], taxi_data_processed['h3_index_dropoff_8']]))
unique_h3_indices_7 = pd.unique(pd.concat([taxi_data_processed['h3_index_pickup_7'], taxi_data_processed['h3_index_dropoff_7']]))

# Create a complete grid of all combinations of hourly timestamps and unique H3 indices
complete_grid_8 = pd.MultiIndex.from_product(
    [hourly_timestamps, unique_h3_indices_8], names=['hour', 'h3_index_8']).to_frame(index=False)

complete_grid_7 = pd.MultiIndex.from_product(
    [hourly_timestamps, unique_h3_indices_7], names=['hour', 'h3_index_7']).to_frame(index=False)

/var/folders/mb/d7qxmv150dz1wszhty3l250c0000gn/T/ipykernel_68057/3426421051.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  taxi_data_processed['Pickup Hour'] = pd.to_datetime(taxi_data_processed['Trip Start Timestamp']).dt.floor('h')
/var/folders/mb/d7qxmv150dz1wszhty3l250c0000gn/T/ipykernel_68057/3426421051.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  taxi_data_processed['Dropoff Hour'] = pd.to_datetime(taxi_data_processed['Trip End Timestamp']).dt.floor('h')


# Resolution 7

In [4]:
# Define aggregations specifically for the Pickups
pickup_aggregations = {
    'Total_Trip_Start': ('Trip ID', 'count'),
    'Unique Taxis': ('Taxi ID', 'nunique'),
    'AvgTripSeconds': ('Trip Seconds', 'mean'),
    'AvgTripMiles': ('Trip Miles', 'mean'),
    'AvgFare': ('Trip Total', 'mean'),
    'MostCommonCompany': ('Company', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    'CompanyCount': ('Company', 'nunique'),
    # TODO: Mean might not be the best way to aggregate lat/lon values, consider using the centroid of the H3 cell instead
    'PickupLongitude': ('Pickup Centroid Longitude', 'mean'),
    'PickupLatitude': ('Pickup Centroid Latitude', 'mean'), 
}

# Define aggregations specifically for the Dropoffs
dropoff_aggregations = {
    'Total_Trip_End': ('Trip ID', 'count')
}

# Perform two separate groupbys
agg_pickups_7 = taxi_data_processed.groupby(
    ['h3_index_pickup_7', 'Pickup Hour']
).agg(**pickup_aggregations).reset_index()

agg_dropoffs_7 = taxi_data_processed.groupby(
    ['h3_index_dropoff_7', 'Dropoff Hour']
).agg(**dropoff_aggregations).reset_index()


# Merge the Pickup data into the complete grid
aggregated_grid_7 = complete_grid_7.merge(
    agg_pickups_7, 
    left_on=['hour', 'h3_index_7'], 
    right_on=['Pickup Hour', 'h3_index_pickup_7'], 
    how='left'
).drop(columns=['Pickup Hour', 'h3_index_pickup_7'])

# Merge the Dropoff data into the complete grid
aggregated_grid_7 = aggregated_grid_7.merge(
    agg_dropoffs_7, 
    left_on=['hour', 'h3_index_7'], 
    right_on=['Dropoff Hour', 'h3_index_dropoff_7'], 
    how='left'
).drop(columns=['Dropoff Hour', 'h3_index_dropoff_7'])

# Fill missing values with 0 for count columns
aggregated_grid_7['Total_Trip_Start'] = aggregated_grid_7['Total_Trip_Start'].fillna(0)
aggregated_grid_7['Total_Trip_End'] = aggregated_grid_7['Total_Trip_End'].fillna(0)
aggregated_grid_7['Unique Taxis'] = aggregated_grid_7['Unique Taxis'].fillna(0)
aggregated_grid_7['AvgTripSeconds'] = aggregated_grid_7['AvgTripSeconds'].fillna(0)
aggregated_grid_7['AvgTripMiles'] = aggregated_grid_7['AvgTripMiles'].fillna(0)
aggregated_grid_7['AvgFare'] = aggregated_grid_7['AvgFare'].fillna(0)
aggregated_grid_7['CompanyCount'] = aggregated_grid_7['CompanyCount'].fillna(0)
aggregated_grid_7['MostCommonCompany'] = aggregated_grid_7['MostCommonCompany'].fillna("")

# Resolution 8

In [5]:
# Define aggregations specifically for the Pickups
pickup_aggregations = {
    'Total_Trip_Start': ('Trip ID', 'count'),
    'Unique Taxis': ('Taxi ID', 'nunique'),
    'AvgTripSeconds': ('Trip Seconds', 'mean'),
    'AvgTripMiles': ('Trip Miles', 'mean'),
    'AvgFare': ('Trip Total', 'mean'),
    'MostCommonCompany': ('Company', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    'CompanyCount': ('Company', 'nunique'),
    # TODO: Mean might not be the best way to aggregate lat/lon values, consider using the centroid of the H3 cell instead
    'PickupLongitude': ('Pickup Centroid Longitude', 'mean'),
    'PickupLatitude': ('Pickup Centroid Latitude', 'mean'), 
}

# Define aggregations specifically for the Dropoffs
dropoff_aggregations = {
    'Total_Trip_End': ('Trip ID', 'count')
}

# Perform two separate groupbys
agg_pickups_8 = taxi_data_processed.groupby(
    ['h3_index_pickup_8', 'Pickup Hour']
).agg(**pickup_aggregations).reset_index()

agg_dropoffs_8 = taxi_data_processed.groupby(
    ['h3_index_dropoff_8', 'Dropoff Hour']
).agg(**dropoff_aggregations).reset_index()


# Merge the Pickup data into the complete grid
aggregated_grid_8 = complete_grid_8.merge(
    agg_pickups_8, 
    left_on=['hour', 'h3_index_8'], 
    right_on=['Pickup Hour', 'h3_index_pickup_8'], 
    how='left'
).drop(columns=['Pickup Hour', 'h3_index_pickup_8'])

# Merge the Dropoff data into the complete grid
aggregated_grid_8 = aggregated_grid_8.merge(
    agg_dropoffs_8, 
    left_on=['hour', 'h3_index_8'], 
    right_on=['Dropoff Hour', 'h3_index_dropoff_8'], 
    how='left'
).drop(columns=['Dropoff Hour', 'h3_index_dropoff_8'])

# Fill missing values with 0 for count columns
aggregated_grid_8['Total_Trip_Start'] = aggregated_grid_8['Total_Trip_Start'].fillna(0)
aggregated_grid_8['Total_Trip_End'] = aggregated_grid_8['Total_Trip_End'].fillna(0)
aggregated_grid_8['Unique Taxis'] = aggregated_grid_8['Unique Taxis'].fillna(0)
aggregated_grid_8['AvgTripSeconds'] = aggregated_grid_8['AvgTripSeconds'].fillna(0)
aggregated_grid_8['AvgTripMiles'] = aggregated_grid_8['AvgTripMiles'].fillna(0)
aggregated_grid_8['AvgFare'] = aggregated_grid_8['AvgFare'].fillna(0)
aggregated_grid_8['CompanyCount'] = aggregated_grid_8['CompanyCount'].fillna(0)
aggregated_grid_8['MostCommonCompany'] = aggregated_grid_8['MostCommonCompany'].fillna("")

In [6]:
aggregated_grid_7.to_parquet(
    "../data/processed/aggregated_grid_7.parquet"
)

aggregated_grid_8.to_parquet(
    "../data/processed/aggregated_grid_8.parquet"
)